### Predicting for the House Price Challenge



### Housing regression project
We trained scikit learn regression models to predict housing sale prices. This notebook retains the assigned sections: OverallQual baseline, missing values, ordinal encoding, one-hot encoding, and pipelines. The final section reproduces the best model actually submitted to Kaggle.

The updated data contains 1,200 training houses and 260 test houses. `SalePrice` is the target, and identifiers are excluded from predictors. Put `train.csv`, `test.csv`, and `sample_soln.csv` alongside this notebook, or upload them through the Colab Files sidebar. Run all cells in order. No `solns.csv` or test prices are used.

Lower dollar RMSE is better. We used five-fold cross-validation to compare models, fitting preprocessing independently inside each fold. Training residuals in the introductory section are not a held-out evaluation. The final submitted model was selected because it has the lowest confirmed Kaggle RMSE among our submissions, even though a later model had a lower validation score.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# In Colab, upload train.csv, test.csv, sample_soln.csv, soln.csv and using the Files sidebar
# Or run these two lines in a separate cell:
# from google.colab import files
# files.upload()

In [ ]:
# Use the updated CSV downloads. Extract a ZIP only if CSVs are absent
import zipfile
DATA_DIR = Path('.')
if not (DATA_DIR / 'train.csv').exists():
    downloads = Path.home() / 'Downloads'
    if (downloads / 'train.csv').exists():
        DATA_DIR = downloads
    elif Path('regression-for-house-prices.zip').exists():
        with zipfile.ZipFile('regression-for-house-prices.zip') as z:
            for name in ['train.csv', 'test.csv']:
                (DATA_DIR / name).write_bytes(z.read(name))
    else:
        raise FileNotFoundError('Upload the updated train.csv and test.csv into this notebook folder')

In [ ]:
df = pd.read_csv(DATA_DIR / 'train.csv')
print('Training rows:', len(df))

In [ ]:
X = df[['OverallQual']]
y = df['SalePrice']

In [ ]:
model = LinearRegression().fit(X, y)

In [ ]:
test = pd.read_csv(DATA_DIR / 'test.csv')
assert 'SalePrice' not in test
assert df.drop(columns='SalePrice').columns.tolist() == test.columns.tolist()
print('Test rows:', len(test))

In [ ]:
preds = model.predict(test[['OverallQual']])

In [ ]:
# Start a submission dataframe for the OverallQual baseline.
pdf = pd.DataFrame({'Id': test.index, 'SalePrice': preds})

In [ ]:
# The current sample uses positional IDs (0, 1, ...), unlike test property IDs.
sample_path = DATA_DIR / 'sample_soln.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path)
    assert len(sample) == len(test) and sample['Id'].is_unique
    pdf['Id'] = sample['Id'].to_numpy()
# Kaggle requires SalePrice, although the sample calls its price column sample_soln.

In [ ]:
# This is the simple baseline, not the final selected model
pdf.to_csv(DATA_DIR / 'baseline_preds.csv', index=False)

In [ ]:
pdf.head()

In [ ]:
df.head()

### Missing Values and Imputation

In [ ]:
df.columns[df.isna().sum() > 0]

In [ ]:
df[df.columns[df.isna().sum() > 0]].isna().sum()/len(df)

### Impute Missing Values

- Fill numeric values with mean
- Fill categorical values with most frequent value

In [ ]:
missing = df[df.columns[df.isna().sum() > 0]]

In [ ]:
nums_missing = missing.select_dtypes(['float', 'int']).columns.tolist()

In [ ]:
cat_missing = missing.select_dtypes('object').columns.tolist()

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
from sklearn.compose import make_column_transformer

In [ ]:
imputer = make_column_transformer((SimpleImputer(), nums_missing),
                                  (SimpleImputer(strategy = "most_frequent"), cat_missing),
                                  remainder = 'passthrough',
                                  verbose_feature_names_out=False)

In [ ]:
imputer.fit_transform(df.drop(columns=['Id', 'SalePrice']))

### Ordinal Encoding

Categorical columns that are ordered can be encoded with the `OrdinalEncoder`.

In [ ]:
df.head()

In [ ]:
df['KitchenQual'].value_counts()

In [ ]:
df['KitchenQual'].value_counts().index.tolist()

Create a list of categories for the encoder.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Quality goes from poor to excellent; TA belongs below Gd.
vals = [['Po', 'Fa', 'TA', 'Gd', 'Ex']]

In [ ]:
oe = OrdinalEncoder(categories=vals)

In [ ]:
oe.fit_transform(df[['KitchenQual']])

In [ ]:
oe.categories_

In [ ]:
oe.transform(df[['KitchenQual']])

In [ ]:
df['KitchenQual'].head()

### OneHotEncoder

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ord_cols = [col for col in [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
] if col in df.columns]

In [ ]:
# Keep one copy of each ordinal column.
ord_cols = list(dict.fromkeys(ord_cols))

In [ ]:
ohe_cols = [col for col in df.select_dtypes(include='object').columns
            if col not in ord_cols]
num_cols = [col for col in df.select_dtypes(include='number').columns
            if col not in ['Id', 'SalePrice']]

In [ ]:
ohe_cols

### Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
vals*len(ord_cols)

In [ ]:
# Each feature belongs to exactly one branch. Each branch imputes before encoding.
imputer = make_column_transformer(
    (SimpleImputer(strategy='median'), num_cols),
    (Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('encode', OrdinalEncoder(categories=[['Missing'] + vals[0]] * len(ord_cols),
                                  handle_unknown='use_encoded_value', unknown_value=-1))
    ]), ord_cols),
    (Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))
    ]), ohe_cols),
    remainder='drop')

In [ ]:
cols = num_cols + ord_cols + ohe_cols
assert len(cols) == len(set(cols))
assert 'SalePrice' not in cols and 'Id' not in cols

In [ ]:
X = df[cols]
y = df['SalePrice']

In [ ]:
cols

In [ ]:
pipe = Pipeline([('fill_and_order', imputer), ('model', LinearRegression())])

In [ ]:
pipe.fit(X, y)

### Separate the encoding and onehot transformations into pipelines

In [ ]:
cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
ord_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
ohe = OneHotEncoder(handle_unknown='ignore')
ord = OrdinalEncoder(categories=[['Missing'] + vals[0]] * len(ord_cols),
                     handle_unknown='use_encoded_value', unknown_value=-1)

ohe_pipe = Pipeline([('impute', cat_imputer), ('ohe', ohe)])
ord_pipe = Pipeline([('impute', ord_imputer), ('ord', ord)])
transformer = make_column_transformer(
    (ohe_pipe, ohe_cols),
    (ord_pipe, ord_cols),
    (SimpleImputer(strategy='median'), num_cols),
    remainder='drop')

In [ ]:
transformer.fit_transform(X)









### Pipeline with Model

In [ ]:
model_pipeline = Pipeline([('transform', transformer), ('model', LinearRegression())])

In [ ]:
model_pipeline.fit(X, y)

In [ ]:
test.head()

In [ ]:
X_test = test[X.columns.tolist()]

In [ ]:
pipeline_preds = model_pipeline.predict(X_test)
print('Pipeline predictions:', len(pipeline_preds))

In [ ]:
# Training residuals are descriptive, not a held-out RMSE estimate.
(y - model_pipeline.predict(X)).hist()

In [ ]:
# The sections below extend the assigned workflow to select by validation RMSE.

### Final submitted model: preprocessing and regression
The final model combines 70% Huber gradient boosting trained on log prices and 30% extra trees trained on dollar prices. Each predicts in dollars before averaging. These models learn nonlinear combinations of house characteristics; averaging can reduce their different errors.

Numeric gaps are median-imputed and numeric features are standardized. Nominal categories are one-hot encoded with unseen categories ignored. Quality ratings are ordered Po < Fa < TA < Gd < Ex; missing quality receives a separate zero code. `MSSubClass` is treated as categorical. Added features include total square footage, bathrooms, house age, and remodel age. All learned preprocessing stays inside cross-validation pipelines.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.model_selection import KFold, cross_validate
train = df.copy()
assert (train['SalePrice'] > 0).all() and test['Id'].is_unique

In [ ]:
def make_features(frame):
    x = frame.drop(columns=['SalePrice', 'Id', 'Order', 'PID'], errors='ignore').copy()
    if 'MSSubClass' in x:
        x['MSSubClass'] = x['MSSubClass'].astype(str)
    quality = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    for col in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']:
        if col in x:
            x[col] = x[col].map(quality).fillna(0)
    x['TotalSF'] = x['TotalBsmtSF'] + x['1stFlrSF'] + x['2ndFlrSF']
    x['TotalBath'] = x['FullBath'] + .5*x['HalfBath'] + x['BsmtFullBath'] + .5*x['BsmtHalfBath']
    x['HouseAge'] = x['YrSold'] - x['YearBuilt']
    x['RemodelAge'] = x['YrSold'] - x['YearRemodAdd']
    return x

X = make_features(train)
y = train['SalePrice']
X_test = make_features(test)
assert (y > 0).all()
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                         ('scale', StandardScaler())])
category_pipe = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
                          ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('numeric', numeric_pipe, num_cols),
                                 ('categorical', category_pipe, cat_cols)])
print('Numeric features:', len(num_cols), 'Categorical features:', len(cat_cols))
print('Most frequent missing columns:')
print(train.isna().mean().sort_values(ascending=False).head(10))

In [ ]:
def log_model(estimator):
    return TransformedTargetRegressor(
        regressor=Pipeline([('preprocess', clone(preprocessor)), ('model', estimator)]),
        func=np.log1p, inverse_func=np.expm1)

# Reproduce the two submitted model approaches compared below.
ridge_model = log_model(Ridge(alpha=10))
boost_model = log_model(GradientBoostingRegressor(
    n_estimators=800, learning_rate=.025, max_depth=3,
    min_samples_leaf=5, loss='huber', random_state=42))
extra_model = Pipeline([('preprocess', clone(preprocessor)),
    ('model', ExtraTreesRegressor(n_estimators=500, max_features=.8,
                                 min_samples_leaf=2, random_state=42, n_jobs=2))])
submitted_tree_model = VotingRegressor(
    [('boost', boost_model), ('extra', extra_model)], weights=[.7, .3])

models = {'Original Ridge submission': ridge_model,
          'Best submitted tree blend': submitted_tree_model}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, candidate in models.items():
    scores = cross_validate(candidate, X, y, cv=cv,
        scoring='neg_root_mean_squared_error', n_jobs=1, error_score='raise')
    rows.append({'Model': name, 'Validation_RMSE': -scores['test_score'].mean()})
results = pd.DataFrame(rows).sort_values('Validation_RMSE')
print(results.to_string(index=False))
# This tree blend has the lowest confirmed Kaggle RMSE among our submissions.
final_model = clone(submitted_tree_model).fit(X, y)


### Export the best confirmed submission
Fit on all training rows and predict the test rows. The downloaded template uses IDs 0–259, whereas test property IDs are 1201–1460. Preserve template IDs in test-row order. Kaggle requires `SalePrice`, although the sample calls the prediction column `sample_soln`. `baseline_preds.csv` from the first section is only the introductory baseline.

In [ ]:
predictions = final_model.predict(X_test)
assert np.isfinite(predictions).all() and (predictions > 0).all()
sample = pd.read_csv(DATA_DIR / 'sample_soln.csv')
assert len(sample) == len(test) and sample['Id'].is_unique
# The competition template uses positional IDs rather than property IDs.
assert sample['Id'].tolist() == list(range(len(test)))
submission = pd.DataFrame({'Id': sample['Id'], 'SalePrice': predictions})
submission.to_csv(DATA_DIR / 'submission_improved.csv', index=False)
assert submission.columns.tolist() == ['Id', 'SalePrice']
assert len(submission) == len(test) and submission.notna().all().all()
print(submission.head().to_string(index=False))
print('Saved submission_improved.csv:', submission.shape)


### Kaggle submissions and reflection

| Submitted version | Kaggle RMSE |
| --- | ---: |
| Original Ridge submission | 161,049.15525 |
| Improved tree blend (final selected model) | **36,105.15526** |
| V4 regression blend | 59,376.14456 |

The improved tree blend has the lowest confirmed Kaggle RMSE. Its fivefold mean validation RMSE was 27,924.49 versus with 28,129.39 for Ridge. The submitted V4 experiment combined  ElasticNet regression with the earlier regression/tree blend. It achieved a lower mean validation RMSE (25,417.15), but the Kaggle score was worse, so we retained the improved tree blend as the final model.

Large and unsual houses  affect RMSE. Ridge made one prediction almost $2 million and the selected tree blend predicted about $531,000 for that property. This is a possible contributor to the difference but not proof of its full cause. Repeated tuning on the same folds also makes validation  optimistic. Kaggle evaluates different houses so the validation gains do not guarantee leaderboard gains. A larger, more representative validation design would be useful future work.